# 10 - Reducción de dimensionalidad y perfiles territoriales

- Unidad de análisis: barrio (48 unidades).
- PCA resume relaciones lineales entre indicadores estandarizados; MCA explora asociaciones entre categorías.
- Las representaciones sirven para visualizar y preparar segmentaciones, no para construir un índice habitacional oficial.
- El PCA/MCA inicial se limita a servicios y transporte: no incluye precios ni alquileres, para que la segmentación posterior no dependa de las variables que el modelo explicativo estudiará.
- Las variables se eligen por cobertura y relevancia para las decisiones; precios con baja cobertura se describen aparte.
- Salidas: varianza, cargas y coordenadas en `data/processed/reports/`.

## 1. Preparación de variables

- Se parte del perfil territorial exportado por 09, construido con los KPIs de servicios y transporte por barrio.
- Se reportan faltantes antes de seleccionar variables; no se imputan variables con cobertura demasiado baja.
- Se aplica log(1+x) a densidades sesgadas seleccionadas y luego estandarización.
- La selección final de variables debe revisarse antes de interpretar componentes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from src import config

REPORTS = config.REPORTS_DIR
perfil = pd.read_csv(config.PROCESSED_DIR / "pe3_features_barrios_v1.csv")
candidatas = [c for c in perfil.columns if c != "barrio_oficial"]
faltantes = perfil[candidatas].isna().mean().sort_values(ascending=False)
display((100 * faltantes).round(1).rename("faltantes_pct").to_frame())

features = [c for c in candidatas if faltantes[c] <= .25]
assert len(features) >= 3, "No hay suficientes variables con cobertura adecuada."
x = perfil[features].replace([np.inf, -np.inf], np.nan).copy()
for col in features:
    if col.endswith("_km2"):
        x[col] = np.log1p(x[col].clip(lower=0))

imputador = SimpleImputer(strategy="median")
escalador = StandardScaler()
x_imp = imputador.fit_transform(x)
x_std = escalador.fit_transform(x_imp)
print("Barrios:", len(perfil), "| variables retenidas:", len(features))
print("Variables:", features)

## 2. PCA: componentes y cargas

- Se presenta varianza explicada por componente y acumulada.
- Las cargas muestran qué variables contribuyen a cada eje.
- Se interpreta cada componente por sus variables con mayor carga absoluta, no por una etiqueta automática.

In [ ]:
pca = PCA()
coord = pca.fit_transform(x_std)
nombres_pc = ["PC{}".format(i + 1) for i in range(coord.shape[1])]
varianza = pd.DataFrame({
    "componente": nombres_pc,
    "varianza_explicada": pca.explained_variance_ratio_,
    "varianza_acumulada": np.cumsum(pca.explained_variance_ratio_),
})
cargas = pd.DataFrame(pca.components_.T, index=features, columns=nombres_pc)
scores = pd.DataFrame(coord, columns=nombres_pc)
scores.insert(0, "barrio_oficial", perfil["barrio_oficial"])

display(varianza.round(3))
display(cargas.iloc[:, :min(3, len(cargas.columns))].round(3))
varianza.to_csv(REPORTS / "pe3_pca_varianza_v1.csv", index=False)
cargas.rename_axis("variable").to_csv(REPORTS / "pe3_pca_cargas_v1.csv")
scores.to_csv(config.PROCESSED_DIR / "pe3_pca_barrios_scores_v1.csv", index=False)

fig, ax = plt.subplots(figsize=(6.5, 3.8))
barras = ax.bar(varianza["componente"], 100 * varianza["varianza_explicada"], color="#5b8db8")
linea, = ax.plot(varianza["componente"], 100 * varianza["varianza_acumulada"], color="#c45a46", marker="o")
ax.set_ylabel("% de varianza")
ax.set_title("PCA por barrio: varianza explicada y acumulada")
ax.legend([linea, barras], ["Acumulada", "Por componente"], frameon=False)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 5.2))
ax.scatter(scores["PC1"], scores["PC2"], color="#4b78a8", s=40, edgecolor="white")
extremos = pd.concat([scores.nsmallest(4, "PC1"), scores.nlargest(4, "PC1"),
                      scores.nsmallest(2, "PC2"), scores.nlargest(2, "PC2")]).drop_duplicates("barrio_oficial")
for _, fila in extremos.iterrows():
    ax.annotate(fila["barrio_oficial"], (fila["PC1"], fila["PC2"]),
                xytext=(4, 3), textcoords="offset points", fontsize=7)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Barrios en el espacio de los dos primeros componentes")
plt.tight_layout()
plt.show()

## 3. MCA: categorías territoriales

- Se discretizan cuatro indicadores en terciles; se reducen automáticamente los cortes si hay empates.
- MCA se usa como contraste cualitativo de perfiles, no para recuperar magnitudes precisas.
- Las categorías se conservan visibles para explicar la posición de cada barrio.

In [ ]:
from scipy.linalg import svd

variables_mca = [
    "densidad_subte_km2", "densidad_metrobus_km2",
    "densidad_culturales_km2", "densidad_educacion_km2",
    "pct_superficie_verde",
]
variables_mca = [c for c in variables_mca if c in perfil.columns]
mca_cat = pd.DataFrame(index=perfil.index)
for col in variables_mca:
    s = perfil[col].replace([np.inf, -np.inf], np.nan)
    if s.notna().sum() < 30 or s.nunique(dropna=True) < 3:
        continue
    mca_cat[col + "_nivel"] = pd.qcut(s, q=3, duplicates="drop").astype(object)
mca_cat = mca_cat.dropna()
assert mca_cat.shape[1] >= 2 and len(mca_cat) >= 30, "No hay cobertura suficiente para MCA."
Z = pd.get_dummies(mca_cat, dtype=float).to_numpy()
P = Z / Z.sum()
masa_fila = P.sum(axis=1)
masa_col = P.sum(axis=0)
den = np.sqrt(np.outer(masa_fila, masa_col))
S = (P - np.outer(masa_fila, masa_col)) / den
U, valores_singulares, Vt = svd(S, full_matrices=False)
coord_mca = U[:, :2] * valores_singulares[:2]
inercia = valores_singulares ** 2
inercia_pct = inercia / inercia.sum()

mca_scores = pd.DataFrame(coord_mca, columns=["Dim1", "Dim2"], index=mca_cat.index)
mca_scores.insert(0, "barrio_oficial", perfil.loc[mca_scores.index, "barrio_oficial"])
display(pd.DataFrame({"dimension": ["Dim1", "Dim2"], "inercia_explicada": inercia_pct[:2]}).round(3))
mca_scores.to_csv(config.PROCESSED_DIR / "pe3_mca_barrios_scores_v1.csv", index=False)

fig, ax = plt.subplots(figsize=(6.5, 4.8))
ax.scatter(mca_scores["Dim1"], mca_scores["Dim2"], color="#6b8f71", s=38, edgecolor="white")
for _, fila in mca_scores.iterrows():
    ax.annotate(fila["barrio_oficial"], (fila["Dim1"], fila["Dim2"]),
                xytext=(3, 2), textcoords="offset points", fontsize=6)
ax.set_xlabel("Dimensión 1")
ax.set_ylabel("Dimensión 2")
ax.set_title("MCA exploratorio de perfiles categorizados por barrio (n = {})".format(len(mca_scores)))
plt.tight_layout()
plt.show()

## 4. Criterios de interpretación

- Revisar si la varianza explicada se concentra en pocos componentes.
- Identificar variables con cargas altas y de signo opuesto.
- Comparar la configuración PCA/MCA con los KPIs originales y con los resultados de 08.
- Si los ejes dependen de pocos casos o cambian mucho al quitar una variable, reducir la interpretación.
- No traducir una proximidad en el gráfico a similitud causal ni a recomendación pública.